[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    B, C_in, H, W = x.size()
    C_out, C_in, kH, kW = weight.size()
    if padding > 0:
        x = F.pad(x, [padding, padding, padding, padding])

    patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
    # [B, C_in, H_out, W_out, kH, kW]
    H_out, W_out = patches.shape[2], patches.shape[3]

    patches = patches.permute(0, 1, 4, 5, 2, 3).contiguous()  # ✅ 加了 permute
    # [B, C_in, kH, kW, H_out, W_out]

    patches = patches.reshape(B, C_in * kH * kW, H_out * W_out)
    # [B, C_in*kH*kW, H_out*W_out]

    weight_matrix = weight.reshape(C_out, -1)
    # [C_out, C_in*kH*kW]

    output = weight_matrix @ patches
    # [B, C_out, H_out*W_out]

    output = output.reshape(B, C_out, H_out, W_out)

    if bias is not None:
        output += bias.view(1, -1, 1, 1)

    return output


In [7]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.6ms)
  ✅ [2/5] Matches F.conv2d (2.7ms)
  ✅ [3/5] With padding (1.2ms)
  ✅ [4/5] With stride (1.0ms)
  ✅ [5/5] Gradient flow (0.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (7.3ms total)
  Progress saved. Run status() to see your dashboard.

